# Interactive polyurethane foam in VR

This tutorial starts a LAMMPS simulation of a segmented polyurethane melt up to
NanoVer, so you can watch a closed foam pore form and manipulate the polymer chains as it does it.

## The system

100 chains of segmented polyurethane -- 15,700 atoms in a 200 x 200 x 200 A
periodic box, using the PCFF (`class2`) force field. Each chain alternates
between two kinds of block:

| segment | chemistry | character |
| --- | --- | --- |
| **Hard** | MDI / urethane | rigid, aromatic |
| **Soft** | PTMO-like polyether | flexible |

That alternation is what gives polyurethane its properties and can be coloured in VR using this notebook to
differentiate the segments.

## What you will see

Two (invisible) spherical walls squeeze the from both sides:

| wall | starts at | ends at |
| --- | --- | --- |
| outer shell (shrinking) | 150 A | 40 A |
| inner bubble (growing) | 1 A | 25 A |

What is left is a a single closed foam pore.

The simulation below **resumes from step 5000**, which has already been run
offline. That first stretch is the part where the outer wall is still closing on
empty space and nothing visible happens, so skipping it means the compaction
starts within seconds of connecting - with the aim of a rooughly 3 minute 'trajectory'. The walls finish moving at step 10,000
and the pore stays formed.

The protocol follows Lavazza et al., *Compression across scales in rigid
polyurethane foams* -- the PDF is in `inputs/`.

## Before you start

Everything the tutorial reads lives in the `inputs/` folder next to this
notebook, and the notebook changes into it before it starts LAMMPS. 
**`inputs/A_VR_3_system_step5000.data`** -- is the checkpoint VR starts from.

If it is missing, or you want to regenerate it, run the production script from
inside `inputs/`. It takes about 3 minutes on 8 cores:

```
cd inputs
mpiexec -n 8 ..\..\..\.venv\Scripts\lmp.exe -in A_inputE1_VR_Prod.in -log none
```

It reads `multiChainsNew.data` and `multiChains.params` from the same folder and
writes nothing but the checkpoint back into it -- no trajectories or logs.

In [ ]:
import os

from nanover.app.omni import OmniRunner

from nanover_extensions.lammps.simulation import LAMMPSSimulation

# The LAMMPS scripts open their files by bare name, so run from inputs/.
os.chdir("inputs")

## Start the simulation server

`frame_interval_steps=5` sends a frame to the client every 5 MD steps, which
works out at roughly 7 frames a second. Lower it for smoother motion, raise it
to advance the chemistry faster at the cost of choppier playback.

Running the cell below starts the server and prints the address to connect to.

In [ ]:
sim = LAMMPSSimulation("A_inputEq_VR_Input.in", frame_interval_steps=5)

runner = OmniRunner.with_basic_server(sim, name="Polyurethane foam")
runner.load(0)

runner.print_basic_info()

Serving "Polyurethane foam" (ws://localhost:38801), discoverable on all interfaces on port 54545
Available simulations:
[0]: "A_inputEq_VR_Input"
Switched to [0]: "A_inputEq_VR_Input"


## Colour the hard and soft segments

By default every atom is coloured by element, which makes the chains hard to tell
apart. The cell below splits the system into hard and soft segments and colours
them the way the source paper does -- dark blue for hard, light blue for soft.

Atoms are classified by LAMMPS atom type. The five types listed below belong to
the MDI/urethane chemistry and everything else is treated as soft. This is a
deliberate simplification: each hard segment contains its own short aliphatic
linker that PCFF types identically to the soft-segment repeat unit, so atom type
alone cannot separate them. It comes out at roughly 13% hard by atom count.

In [3]:
import numpy as np
from nanover.websocket import NanoverImdClient

# Types unique to the MDI/urethane hard-segment chemistry (see multiChains.params):
# c_2 (3, carbamate carbon), cp (4, aromatic carbon), hn (6, N-H hydrogen),
# na (7, urethane nitrogen), o_1 (8, carbonyl oxygen).
HARD_TYPES = {3, 4, 6, 7, 8}

types = np.asarray(sim.lmp.gather_atoms("type", 0, 1), dtype=np.int32)
is_hard = np.isin(types, list(HARD_TYPES))
hard_ids = np.nonzero(is_hard)[0].tolist()
soft_ids = np.nonzero(~is_hard)[0].tolist()

print(
    f"{len(hard_ids):,} hard atoms ({len(hard_ids) / len(types):.1%}), {len(soft_ids):,} soft atoms"
)

HARD_COLOR = "#1B3A6B"  # dark blue, matches the paper's hard-segment convention
SOFT_COLOR = "#9FC5E8"  # light blue, matches the paper's soft-segment convention

with NanoverImdClient.from_runner(runner) as client:
    with client.create_selection("Hard segment", hard_ids).modify() as selection:
        selection.renderer = {
            "color": {
                "type": "cpk",
                "scheme": {"C": HARD_COLOR, "N": HARD_COLOR, "O": HARD_COLOR, "H": HARD_COLOR},
            }
        }
    with client.create_selection("Soft segment", soft_ids).modify() as selection:
        selection.renderer = {
            "color": {
                "type": "cpk",
                "scheme": {"C": SOFT_COLOR, "O": SOFT_COLOR, "H": SOFT_COLOR},
            }
        }

2,000 hard atoms (12.7%), 13,700 soft atoms


## Things to try in VR

- **Watch the pore close.** 
- **Pull a chain out of the wall.** Grab an atom and drag it away from the pore.
- **Compare hard and soft.** The dark blue MDI units might feels like they resit being pulled more.



## Shut down

Run this to tidy up.

In [1]:
runner.close()

NameError: name 'runner' is not defined